# LAB 09 — From Dev to Prod with Declarative Automation Bundles

In class, the trainer deploys this bundle on the shared screen, and you answer the quiz **Lab 09 — From dev to prod with bundles (quiz)** in the Workshop Guide app (in the Polish version: **Lab 09 — Z dev na prod z bundle (quiz)**). After the course, do the tasks yourself and deploy the bundle in the web terminal of this notebook (Task 3, Option A).

**Goal:** Deploy the RetailHub capstone to the `dev` target as a **Declarative Automation Bundle** (formerly *Databricks Asset Bundles*, DABs), and check the deployment with code. The bundle holds a Lakeflow Spark Declarative Pipeline and a Lakeflow Job with three tasks.

**Prerequisites:**
- lab_07 is done, so you know the pipeline sources in `materials/lakeflow/lakeflow_demo`.
- Tasks 3 and 5 need the Databricks CLI. The simplest is the **web terminal** of this notebook on Serverless (Task 3, Option A): the CLI and your login are already there. The local CLI on your laptop also works (Option B).
- Only the optional Git task and Option C need the repo as a **Git folder** in your workspace.

| Task | What you do | Where |
|------|-------------|-------|
| 1 | Read `databricks.yml` and describe the bundle | notebook |
| 2 | Build the deploy command with your catalog variable | notebook |
| *opt.* | In the Git folder, create a branch, commit, push and open a pull request (depends on the trainer) | UI |
| 3 | Run `databricks bundle validate` and `deploy`, then record how you ran them | terminal (or UI / trainer) + notebook |
| 4 | Find the deployed job with the Databricks SDK | notebook |
| 5 | Run the job and check that it ends with SUCCESS | terminal or UI + notebook |
| 6 | Reflection: compare the `dev` and `prod` targets | notebook |

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# PROVIDED (run as it is): find the bundle root and print databricks.yml
import os

# The bundle lives in materials/cicd/ at the repo root. Walk up from this notebook's folder
# until materials/cicd/databricks.yml is found — works from notebooks/day3/lab/ and notebooks/solution/.
def find_bundle_dir(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        candidate = os.path.join(d, "materials", "cicd")
        if os.path.isfile(os.path.join(candidate, "databricks.yml")):
            return candidate
        parent = os.path.dirname(d)
        if parent == d:          # reached the filesystem root
            return None
        d = parent

BUNDLE_DIR = find_bundle_dir()
print(f"Bundle root: {BUNDLE_DIR}\n")

try:
    if BUNDLE_DIR is None:
        raise FileNotFoundError(f"materials/cicd/databricks.yml not found in {os.getcwd()} or any parent folder")
    databricks_yml = open(os.path.join(BUNDLE_DIR, "databricks.yml")).read()
    print(databricks_yml)
except Exception as e:
    databricks_yml = None
    print("Could not read the file from here — open materials/cicd/databricks.yml in the workspace file browser instead.")
    print(e)

## Task 1 — Inspect `databricks.yml`: bundle anatomy

Every bundle starts from `databricks.yml`. When you get a bundle from a colleague, read this file first: it tells you what the bundle deploys and where.

Read the file printed above, or open `materials/cicd/databricks.yml`. Then fill in the `bundle_structure` dict with the bundle name, the default target, the mode of that target and the names of all declared variables.

In [0]:
# TODO: describe the bundle from databricks.yml (printed in the Setup cell)
bundle_structure = {
    "bundle_name":         ...,   # bundle.name
    "default_target":      ...,   # which target has default: true?
    "default_target_mode": ...,   # that target's mode
    "variable_names":      [...], # names of ALL declared variables
}
print(bundle_structure)

#### Hint — Task 1

The root file `databricks.yml` has four top-level sections. On the exam you must be able to read them without help:

- `bundle:` holds the bundle `name`. Everything the bundle deploys belongs to this name.
- `include:` lists more YAML files that are merged into the config. Here it is `resources/*.yml`, one file per resource.
- `variables:` declares inputs, each with an optional `default`. You refer to a variable as `${var.<name>}` and override it for one deploy with `--var="name=value"`.
- `targets:` declares the environments. Only one target can have `default: true`. `mode: development` gives dev-loop behavior (name prefixes, paused triggers), and `mode: production` is strict.

Also look at `sync.paths`. This bundle uses source folders outside the bundle root
(`../lakeflow/lakeflow_demo`, `../orchestration`), so a deploy from your laptop uploads them too.

In [0]:
# -- Validation --
assert bundle_structure["bundle_name"] == "retailhub", "Check bundle.name in databricks.yml"
assert bundle_structure["default_target"] == "dev", "Which target carries default: true?"
assert bundle_structure["default_target_mode"] == "development", "Check the mode of the dev target"
assert sorted(bundle_structure["variable_names"]) == ["catalog", "schema_prefix", "training_group"], \
    f"Expected the three declared variables, got: {bundle_structure['variable_names']}"
print("Task 1 OK: bundle anatomy understood")

## Task 2 — Compose the deploy command

The `catalog` variable of the bundle defaults to `retailhub_trainer`. If you deploy without an override, the pipeline writes to the trainer's catalog, where you have no rights, and the deploy or the pipeline refresh in Task 5 fails on permissions. In CI/CD you use the same kind of override to deploy one bundle with different values.

Build `deploy_command` as a Python string: a `databricks bundle deploy` command for the `dev` target that sets the `catalog` variable to **your** `CATALOG` value.

In [0]:
# TODO: build the deploy command for your catalog (the CATALOG variable)
# Shape: databricks bundle deploy -t dev --var="catalog=<your catalog>"
deploy_command = ...  # YOUR CODE HERE (f-string)

print(deploy_command)

#### Hint — Task 2

The command has this shape:

```
databricks bundle deploy -t <target> --var="<name>=<value>"
```

- `-t dev` selects the target. `dev` is the default here, but in CI you always name the target.
- `--var="catalog=retailhub_jan_kowalski"` overrides a declared variable for this one command.
  For more variables, repeat the flag.
- `00_setup` already exported your catalog name in the `CATALOG` variable. Use an f-string.

You can also set the variable in the environment (`BUNDLE_VAR_catalog`) or in the `variables:` of a target
in `databricks.yml`. The `--var` flag wins over the defaults.

In [0]:
# -- Validation --
assert isinstance(deploy_command, str), "deploy_command must be a string"
assert deploy_command.startswith("databricks bundle deploy"), "Start with: databricks bundle deploy"
assert "-t dev" in deploy_command, "Target the dev target explicitly: -t dev"
assert f"catalog={CATALOG}" in deploy_command, "Override the catalog variable with YOUR catalog (use CATALOG)"
print(f"Task 2 OK: {deploy_command}")

## Optional — Git folder workflow before deploying (objective 5.1)

> **Depends on the trainer.** You need a Git folder connected to a Git provider where you can push. You set the Git credentials in **Settings → Linked accounts**. If no provider is connected, read the steps and go on to Task 3. No assert checks this part.

A team changes a bundle on a branch, and a colleague reviews the change in a pull request before it goes further.

1. Go to **Workspace** → your Git folder and click the **branch name** to open the Git dialog. Select **Create Branch** and create `feature/<your_name>` from `main`.
2. Open `materials/cicd/resources/retailhub_job.yml` and make a small change that breaks nothing. For example, extend the `description`, or add these lines at the same indentation as `max_concurrent_runs`:
   ```yaml
         tags:
           owner: <your_name>
   ```
3. In the Git dialog, check the diff, enter the commit message `lab09: tag retailhub_job` and select **Commit & Push**.
4. In the Git provider, open a **pull request** from `feature/<your_name>` to `main`. Do not merge it unless the trainer tells you to.
5. Stay on your feature branch. Task 3 deploys **the files of the place you run it from**: the Git folder (web terminal or workspace UI) or your local clone after you pull `feature/<your_name>`. Your change then ends up in the deployed job, which is a small CI/CD loop.


## Task 3 — Validate & deploy the bundle

`validate` checks the bundle and changes nothing in the workspace. `deploy` creates or updates the job and the pipeline. A CI pipeline runs the same two commands.

> **You deploy outside this notebook.** Choose ONE option from the table below and follow it. After the course, Option A is the simplest. In class, the trainer's deploy on the shared screen follows the Option D checklist.
> Then record what you did in the answer cell. The check cell looks for the deployment in your workspace.

| Your situation | Option | Value for `how_i_ran_it` |
|---|---|---|
| This notebook runs on Serverless (the default) | **A**: Web terminal of this notebook | `"web_terminal"` |
| You have the Databricks CLI on your laptop | **B**: Local Databricks CLI | `"local_cli"` |
| `databricks.yml` in your Git folder shows a **Deploy bundle** button | **C**: Workspace UI deploy | `"workspace_ui"` |
| The web terminal does not work for you (and B and C are not possible) | **D**: Trainer-driven fallback | `"trainer"` |

### Option A — Web terminal of this notebook (default)
1. With this notebook on **Serverless**, select **Terminal** in the bottom panel of the notebook. The first command in a new terminal session installs the Databricks CLI (v1.27.0 on the training workspace), which takes a few seconds. The terminal uses your login, so you do not need a `-p` profile.
2. The terminal starts in the folder of this notebook. Go to the folder that the Setup cell printed as **Bundle root**, then validate and deploy. The command that **Task 2** printed works as it is.

```bash
cd "<Bundle root from the Setup cell>"
databricks bundle validate -t dev --var="catalog=<YOUR CATALOG>"
# read the summary: name, target, workspace host, user, path

databricks bundle deploy   -t dev --var="catalog=<YOUR CATALOG>"   # <- Task 2 command
```

A `dev` deploy from the workspace is **source-linked**. The CLI prints `Source-linked deployment is enabled` and `Files: 0 uploaded`, so nothing is copied: the job tasks point at the notebooks in the course folder. The deploy still keeps its state in `/Workspace/Users/<you>/.bundle/retailhub/dev`, and the check cell looks for that folder.

### Option B — Local Databricks CLI (CLI 1.26 or newer, tested with v1.27.0)
1. Install the current Databricks CLI and check it with `databricks -v`. The steps are in `09_cicd_and_bundles` → *Databricks CLI — Setup and Authentication*. Do **not** use `pip install databricks-cli`.
2. Clone the repo on your laptop, on the same branch as your Git folder. Log in once, then validate and deploy from the bundle root.
   Replace `<YOUR CATALOG>` with your catalog, or paste the command that **Task 2** printed and add `-p TRAINING`.
   `validate` takes the same flags.

```bash
databricks auth login --host <workspace-url> --profile TRAINING
cd <your clone>/materials/cicd

databricks bundle validate -t dev --var="catalog=<YOUR CATALOG>" -p TRAINING
# read the summary: name, target, workspace host, user, path

databricks bundle deploy   -t dev --var="catalog=<YOUR CATALOG>" -p TRAINING   # <- Task 2 command (+ profile)
```

A deploy from your laptop uploads the bundle files to `.bundle/retailhub/dev/files` in your home folder.

### Option C — Workspace UI deploy (no CLI)
Open `materials/cicd/databricks.yml` in your Git folder. If the workspace shows a **Deploy bundle** button,
you can validate and deploy from the workspace UI. Make sure that `catalog` resolves to **your** catalog,
and record `"workspace_ui"`. The button needs a Git folder: a `databricks.yml` in a plain workspace folder does not show it.

### Option D — Trainer-driven fallback (the web terminal does not work)
The trainer runs `validate` and `deploy` on the shared screen. Watch for these points:

- [ ] `validate` prints the bundle **name** (`retailhub`) and the **target** (`dev`).
- [ ] `validate` prints the **workspace path** of the deployment (`/Workspace/Users/<user>/.bundle/retailhub/dev`).
- [ ] `validate` ends with **no errors** ("Validation OK").
- [ ] `deploy` creates the pipeline and the job. From the workspace it prints `Source-linked deployment is enabled` and `Files: 0 uploaded`: the job uses the source files where they are.
- [ ] In **Jobs & Pipelines**, the job **`[dev <user>] retailhub_job`** and the pipeline **`[dev <user>] retailhub_pipeline`** appear.
- [ ] The `[dev …]` prefix comes from `mode: development`. `retailhub_job` has no schedule (it is commented out in the YAML), but dev mode would pause any schedule or trigger.

Then do Tasks 4 and 5 on the job that the trainer deployed. You have view access to it: the bundle gives **Can View** on the job and the pipeline to the group in the `training_group` variable (by default `users`, every workspace user).

Task 4 then prints a WARNING: no job has your name, so it uses the first `retailhub_job` it finds, the trainer's or another participant's. That is expected in Option D.

In [0]:
# TODO: record how you deployed the bundle and whether validate passed
task3_result = {
    "how_i_ran_it":       ...,  # "local_cli" | "web_terminal" | "workspace_ui" | "trainer"
    "validate_succeeded": ...,  # did `bundle validate -t dev` finish without errors? True/False
}
print(task3_result)

In [0]:
# -- Validation --
from databricks.sdk import WorkspaceClient

assert task3_result["how_i_ran_it"] in ("local_cli", "web_terminal", "workspace_ui", "trainer"), \
    "Use one of: local_cli, web_terminal, workspace_ui, trainer"
assert task3_result["validate_succeeded"] is True, \
    "bundle validate must pass before deploying — ask the trainer if it failed"

# Real check: a dev-mode deploy writes its files and state under YOUR user home
w = WorkspaceClient()
_user_name = w.current_user.me().user_name
bundle_root_path = f"/Workspace/Users/{_user_name}/.bundle/retailhub/dev"
try:
    w.workspace.get_status(bundle_root_path)
    bundle_deployed, _err = True, None
except Exception as e:   # NotFound when nothing was deployed under your identity
    bundle_deployed, _err = False, e

if task3_result["how_i_ran_it"] == "trainer":
    print(f"Trainer-driven deploy — your own bundle folder {'exists' if bundle_deployed else 'is not expected'}: {bundle_root_path}")
    print("Task 3 OK (trainer-driven): Task 4 looks up the trainer's deployed job")
else:
    assert bundle_deployed, (
        f"No bundle deployment found at {bundle_root_path}. Run `databricks bundle deploy -t dev "
        f'--var="catalog={CATALOG}"` from materials/cicd as the same user as this notebook, then re-run this cell. '
        f"(details: {_err})"
    )
    print(f"Task 3 OK: bundle deployed via {task3_result['how_i_ran_it']} — found {bundle_root_path}")

## Task 4 — Verify the deployment from this notebook

A CI pipeline cannot click through the UI, so it checks a deployment with code. Here you use the Databricks SDK
(`databricks.sdk`, already installed on Databricks compute) to list the jobs and find the deployed
RetailHub job.

Build `retailhub_jobs`: all the jobs you can see whose name contains `retailhub_job`.
`mode: development` deployed the job as **`[dev <user>] retailhub_job`**.

Then run the check cell. It picks `retailhub_job_id`, which Task 5 uses.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# TODO: list all jobs you can see whose name contains "retailhub_job"
retailhub_jobs = []  # YOUR CODE HERE — replace [] with a list comprehension over w.jobs.list(), filtering on j.settings.name

for j in retailhub_jobs:
    print(f"{j.job_id}  {j.settings.name}")

#### Hint — Task 4

Inside a Databricks notebook, `WorkspaceClient()` without arguments logs in from the runtime context,
so you do not pass a host or a token.

To list the jobs:

```python
for j in w.jobs.list():
    print(j.job_id, j.settings.name)
```

`w.jobs.list()` returns an iterator of `BaseJob`, and the display name is in `j.settings.name`.
A list comprehension that tests the name for a substring is all you need.

`mode: development` adds the prefix `[dev <your user name>]` to every resource name, so many people
can deploy the same bundle to one workspace without name collisions. In `mode: production` the name
stays exactly `retailhub_job`.

In [0]:
# -- Validation --
assert len(retailhub_jobs) >= 1, "No deployed retailhub_job found — did Task 3 deploy succeed?"
_names = [j.settings.name for j in retailhub_jobs]
assert any(n.startswith("[dev") and "retailhub_job" in n for n in _names), \
    f"Expected a dev-mode name like '[dev <user>] retailhub_job', got: {_names}"

# Prefer your own copy if several participants deployed into this workspace
_me = w.current_user.me().user_name.split("@")[0].replace(".", "_").lower()
_mine = [j for j in retailhub_jobs if _me in j.settings.name.lower()]
if not _mine:
    print(f"WARNING: no retailhub_job with '{_me}' in its name — falling back to '{retailhub_jobs[0].settings.name}', "
          "which may be another participant's (or the trainer's) deployment")
    _mine = retailhub_jobs
retailhub_job_id = _mine[0].job_id
print(f"Task 4 OK: found {_names} — using job_id={retailhub_job_id}")

## Task 5 — Run the job and confirm it succeeded

A deploy creates or updates the job but does not run it. A run shows that the job works in the target environment.
Start a run of the deployed job, then check with the SDK from this notebook that the run ended
with the result state **SUCCESS**.

Start the run in one of these ways:

```bash
# Databricks CLI: streams task states until the job finishes.
# Web terminal (Task 3, Option A): run it as it is. Local CLI (Option B): add -p TRAINING.
databricks bundle run -t dev --var="catalog=<YOUR CATALOG>" retailhub_job
```

At the end the CLI prints `Output:` lines for the notebook tasks with nothing after them, because the task notebooks return no value. What counts is `TERMINATED SUCCESS`.

- Or select **Jobs & Pipelines → `[dev …] retailhub_job` → Run now**. You may use the UI here, because the lab is about the bundle.
- Or the **trainer** runs the shared deployment, and you check that run.

The run goes through validate → pipeline refresh → report. Run **both** cells below again until `result_state` is no longer `None`.

In [0]:
# TODO: take the latest run of the job and read its result state
from itertools import islice

# list_runs() pages lazily through ALL runs of the job — take only the newest 5 (newest first)
runs = list(islice(w.jobs.list_runs(job_id=retailhub_job_id, limit=5), 5))

latest_run   = ...   # YOUR CODE HERE — the most recent run (None if runs is empty)
result_state = ...   # YOUR CODE HERE — its result state as a string ("SUCCESS", ...)
                     # hint: latest_run.state.result_state, None until the run terminates

print(f"life_cycle_state = {latest_run.state.life_cycle_state.value if latest_run and latest_run.state else '?'}")
print(f"result_state     = {result_state}")

#### Hint — Task 5

`runs` from the cell above is a plain list, **newest first**. Check that the list is not empty before you take an element from it.

A run has two state fields, and the exam often mixes them up:
- `run.state.life_cycle_state` says where the run is: `PENDING → RUNNING → TERMINATED`.
- `run.state.result_state` is **set only after the run ends**: `SUCCESS`, `FAILED`, `CANCELED`, `TIMEDOUT`.

A healthy finished run has `life_cycle_state = TERMINATED` **and** `result_state = SUCCESS`.
Both fields are enums. Use `.value` to get the string, and handle `None` while the run is still going.

In [0]:
# -- Validation --
assert len(runs) >= 1, "No runs found — trigger the job first (bundle run / Run now / trainer)"
assert result_state is not None, \
    "Run has not reached a terminal state yet — wait a bit and re-run the previous cell + this one"
assert str(result_state).upper() == "SUCCESS", \
    f"Run terminated with {result_state} — open the run in Jobs & Pipelines and check the failed task"
print(f"Task 5 OK: run {latest_run.run_id} finished with result_state=SUCCESS")

## Task 6 — Reflection: `dev` vs `prod` targets

Before you promote a bundle to production, you need to know what changes between the targets and what stays the same.
Look at the commented `prod` stub at the end of `databricks.yml` and at what `mode: development`
did to your deployment. Then fill in the `reflection` dict.

> `retailhub_job` has no schedule (it is commented out in `resources/retailhub_job.yml`), so you cannot *see* a paused
> trigger in this deployment. The question is what `mode: development` **would** do to a schedule or a trigger.

In [0]:
# TODO: answer the dev vs prod questions
reflection = {
    "dev_name_prefix":       ...,  # string: what development mode prepends to resource names (look at the job name in Jobs & Pipelines)
    "dev_pauses_triggers":   ...,  # are schedules/triggers paused in a development-mode deployment? True/False
    "prod_runs_as":          ...,  # recommended identity for prod deployments: "personal_user" | "service_principal"
    "promotion_changes":     ...,  # what changes when promoting dev -> prod: "the_code" | "only_target_config"
}
print(reflection)

#### Hint — Task 6

Compare what you saw with what the `prod` stub declares:

| Aspect | `dev` (`mode: development`) | `prod` (`mode: production`) |
|---|---|---|
| Resource names | prefixed | exact names from the YAML |
| Schedules & triggers | paused automatically | active |
| Deploy path | your user home (`/Users/<you>/.bundle/...`) | a restricted folder: the home of the service principal (`run_as`), not `/Workspace/Shared`, where all users can write |
| Identity | you | usually a **service principal** (`run_as`) |
| Code | **the same YAML and sources** | the same, only the target config differs |

The last row is the main idea of CI/CD, and the exam often asks about it: promotion between environments changes
the **target configuration**, never the code.

In [0]:
# -- Validation --
assert "[dev" in str(reflection["dev_name_prefix"]), \
    "Look at your deployed job's name in Jobs & Pipelines"
assert reflection["dev_pauses_triggers"] is True, \
    "development mode pauses schedules/triggers so dev copies never fire on their own"
assert reflection["prod_runs_as"] == "service_principal", \
    "Production deployments should not depend on a person's identity"
assert reflection["promotion_changes"] == "only_target_config", \
    "Same YAML + sources deploy everywhere; only the target section differs"
print("Task 6 OK: dev vs prod understood")

## Common pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `bundle validate` fails with variable errors | A typo in the `--var` flag | Write it exactly as `--var="catalog=<your catalog>"`, as in Task 2 |
| Permission error on the deploy or on the pipeline refresh | No `--var` override, so the pipeline uses the trainer catalog `retailhub_trainer` | Deploy again with the Task 2 command: `--var="catalog=<your catalog>"` |
| `retailhub_jobs` is empty | The deploy ran in another workspace or CLI profile, the filter is too strict, or (Option D) the trainer's deploy gave view access to a group you are not in | Deploy with the profile of this workspace, and filter on the substring `retailhub_job`: development mode adds the prefix `[dev <user>] `. In Option D the trainer deploys again with `--var="training_group=<your group>"` |
| `result_state` is `None` | The run is still `RUNNING`. `life_cycle_state` says where the run is; `result_state` is set only when the run ends | Wait, then run the fetch cell and the check cell of Task 5 again |

## Summary

| Task | Topic | Key point |
|------|-------|-----------|
| 1 | Bundle anatomy | `bundle`, `include`, `variables`, `targets`, and `sync.paths` for sources outside the bundle root |
| 2 | Variables | `${var.catalog}` in the YAML, overridden for one deploy with `--var="catalog=..."` |
| 3 | CLI flow | `validate` → `deploy` → the resources appear. `mode: development` adds the `[dev …]` prefix and pauses triggers |
| 4 | SDK verification | `WorkspaceClient().jobs.list()` checks a deployment with code instead of clicks |
| 5 | Run states | `life_cycle_state` (PENDING/RUNNING/TERMINATED) vs `result_state` (SUCCESS/FAILED/…) |
| 6 | dev vs prod | Promotion changes the target config (identity, paths, variables), never the code |

**Stretch goals (optional):**
- Point the `publish_report` task at a RetailHub gold table and deploy again. You edit `resources/retailhub_job.yml`,
  and the task needs a report notebook that reads RetailHub data (see the comment in the YAML).
- Remove your deployment with `databricks bundle destroy -t dev --var="catalog=<YOUR CATALOG>"` (with the local CLI, add `-p TRAINING`).
  Then check in Jobs & Pipelines that both resources are gone.

> **Exam (Implementing CI/CD):** the Git folder flow (branch, commit, push, pull request), the anatomy of the bundle file,
> `validate`/`deploy`/`run`/`destroy`, targets and modes, variable overrides, and why bundles are better than manual setup
> in the UI: they are versioned, reviewable, repeatable and can be promoted between environments.

← <a href="$../demo/09_cicd_and_bundles">09 — CI/CD & Bundles</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./lab_troubleshooting">Lab — Troubleshooting →</a>